# 第6章 数の表を一気に扱う：テンソル

for文を卒業して、420万マスの表を一瞬で計算する

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：bigramモデルを「確率の表（行列）」として作り直し、損失計算を一瞬で終わらせる
- **身につく数学・理論**：ベクトル・行列・テンソルと「形（shape）」 ／ 行ごとの合計とブロードキャスト ／ one-hotベクトルと行列の掛け算（＝表の行を取り出す） ／ 行列の積の形のルール $(n \times k)(k \times m) = (n \times m)$ ／ なぜGPUは速いのか（並列計算）
- **新しく使うPython・PyTorch**：`torch.tensor` / `torch.zeros` ／ `.shape` と `.dtype` ／ インデックスとスライス（`N[i, j]`、`N[:, j]`） ／ `sum(dim=…, keepdim=True)` ／ `@`（行列の掛け算） ／ `time` で計測

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 60〜90分 |
| 難易度 | ★★☆☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第5章では、Pythonのfor文で1文字ずつ確率を計算しました。それでも動きますが、これから作るモデルは**何百万回、何億回**も計算します。for文では日が暮れて、夜が明けて、また日が暮れます。この章では、たくさんの数をまとめて一気に計算する道具 **テンソル** を手に入れ、bigramモデルを「確率の表」として作り直します。

> **🔗 これまでの復習とつながり：第3章・第4〜5章とのつながり**
>
> - **第3章**：`out.logits.shape` が `[1, 6, 151936]` という3次元の「形」を持ち、`dim=-1` で最後の軸に沿って `softmax` をかけたことを見ました。
> - **第4〜5章**：『こころ』の語彙2061文字から `stoi`（文字→番号）と bigram の回数表（2061×2061＝約420万マス）、スムージング定数 `k=0.01`、検証損失 `3.581` を作りました。
>
> この章では、第3章でちらっと顔を出した **テンソル・`shape`・`dim`** の正体を完全に解き明かし、第4〜5章の bigram 確率表と損失計算を**for文なし・数行・一瞬**で動かせるように書き換えます！

**ミナト**：テンソルって、名前がもう強そうです。

**ボンゴ先生**：名前はね。でも中身は「数を並べた箱」だ。1列に並べたらベクトル、表にしたら行列、それを何枚も重ねたら3次元のテンソル。HTMLの `<table>` が書けるキミなら、もう半分わかっている。

## 準備

いつもの準備セルです（第5章と同じ）。今回は `torch` も使います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, math, time, urllib.request
from collections import Counter
import torch

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print("語彙の数：", V)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
語彙の数： 2061
```

</details>

> **🐍 Python・PyTorchのポイント：辞書を1行で作る**
>
> `{ch: i for i, ch in enumerate(chars)}` は、第2章でfor文を使って作った `stoi` を1行で書いたものです（**辞書内包表記**）。やっていることは同じです。

<details><summary><b>基礎の補足：理解のための最小コード：辞書内包表記 `{ch: i for i, ch in enumerate(chars)}`</b>（クリックで開く）</summary>

| ステップ | 値（具体例） | 型 | 起きていること |
|---|---|---|---|
| 入力 | `chars = ['あ', 'い', 'う']` | `list` of `str` | `sorted(set(text))` で重複なし・順番固定のリスト |
| `enumerate(chars)` | `(0, 'あ'), (1, 'い'), (2, 'う')` | `(int, str)` の列 | 0始まりの番号 `i` と要素 `ch` を組にする |
| 出力 `stoi` | `{'あ': 0, 'い': 1, 'う': 2}` | `dict` (`str -> int`) | 文字をキー、番号を値にした辞書を一気に作る |
| 出力 `itos` | `{0: 'あ', 1: 'い', 2: 'う'}` | `dict` (`int -> str`) | キーと値を入れ替えて逆引き辞書を作る |

普通の `for` 文と完全に対応しています：
```python no-run
chars = ["あ", "い", "う"]
# 通常の for 文で書くと3行：
stoi_loop = {}
for i, ch in enumerate(chars):
    stoi_loop[ch] = i
# 辞書内包表記で書くと1行：
stoi = {ch: i for i, ch in enumerate(chars)}
print(stoi == stoi_loop, stoi)   # True {'あ': 0, 'い': 1, 'う': 2}
```

</details>

#### 📝 確認問題（`ch06-q01` · Lv.2 · 辞書内包表記による stoi / itos の作成）

辞書内包表記で文字→IDの対応表 `stoi` と逆引き表 `itos` を作ります。次のコードを実行すると何が表示されますか？

```python
chars = ["あ", "い", "う"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print(stoi["う"], itos[1])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`enumerate(chars)` は `0` 始まりの番号と文字のペア `(i, ch)` を順に出します。
- **ヒント 2**：`あ:0, い:1, う:2` の対応表 `stoi` と逆引き `itos` を書き出して、`stoi["う"]` と `itos[1]` を確認しましょう。
- **期待される出力**：`2 い`

**考え方**：`stoi` は `{'あ': 0, 'い': 1, 'う': 2}`、`itos` は `{0: 'あ', 1: 'い', 2: 'う'}` となるため、`stoi["う"]` は `2`、`itos[1]` は `'い'` です。

1. `enumerate(chars)` → `(0, 'あ'), (1, 'い'), (2, 'う')`。
1. `stoi` = `{'あ': 0, 'い': 1, 'う': 2}` → `stoi["う"] = 2`。
1. `itos` = `{0: 'あ', 1: 'い', 2: 'う'}` → `itos[1] = 'い'`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q01

</details>

## テンソル入門：数を並べた箱

テンソルを作ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print(a + b)
print(a * b)
print(a * 2)
print(a.sum())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([11., 22., 33.])
tensor([10., 40., 90.])
tensor([2., 4., 6.])
tensor(6.)
```

</details>

ポイントは、**for文を書いていないのに、全部の要素がいっぺんに計算されている**ことです。`a + b` は「同じ位置どうしを足す」、`a * 2` は「全部を2倍」。リストでやろうとすると、for文が必要でしたね。

次は2次元、つまり表です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
M = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print(M.shape)
print(M[0])       # 0行目
print(M[:, 1])    # 1列目（全部の行の、1番目）
print(M[1, 2])    # 1行目の2列目

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2, 3])
tensor([1, 2, 3])
tensor([2, 5])
tensor(6)
```

</details>

#### 💡 理論のポイント：ベクトル・行列・テンソルと「形」

- 数を1列に並べたもの：**ベクトル**（vector）。`a` は長さ3のベクトル
- 数を表（縦×横）に並べたもの：**行列**（matrix）。`M` は2行3列の行列
- さらに何枚も重ねたもの（3次元以上も含めた総称）：**テンソル**（tensor）

| 次元数（ランク） | 呼び名 | 具体例 | `.shape` | 要素数 `.numel()` |
|---|---|---|---|---|
| 0次元 | スカラー（数1個） | `torch.tensor(6)` | `[]` | $1$ |
| 1次元 | ベクトル | `torch.tensor([1, 2, 3])` | `[3]` | $3$ |
| 2次元 | 行列（表） | `M`（2行3列） | `[2, 3]` | $2 \times 3 = 6$ |
| 3次元 | 3次元テンソル | 第3章の `out.logits` | `[1, 6, 151936]` | $1 \times 6 \times 151936$ |

**多次元インデックスの読み方**（すべて **0始まり**、`[行, 列]` の順）：
- `M[0]` → 0行目ぜんぶ：`[1, 2, 3]`（形 `[3]`）
- `M[:, 1]` → 全行の1列目：`[2, 5]`（形 `[2]`。`:` は「その軸を全部」の意味）
- `M[1, 2]` → 1行目の2列目：`6`（形 `[]`）

テンソルを扱うときに一番大事なのは **形**（shape）です。この先、エラーの半分以上は「形が合わない」ことが原因です。迷ったら、とにかく `.shape` を `print` しましょう。これはこの教科書で一番役に立つアドバイスかもしれません。

> **📘 基本情報とのつながり：2次元配列**
>
> 基本情報の擬似言語やアルゴリズムの問題で出てくる「2次元配列」が、まさに行列です。`M[1, 2]` は「1行目の2列目」。ただしPythonは0から数えるので、人間の感覚では「2行目の3列目」です。試験の問題が1から数えるか0から数えるかは、毎回確認しましょう。

#### 📝 確認問題（`ch06-q02` · Lv.1 · スカラー・ベクトル・行列・テンソルと shape の読み方）

次の4つのテンソルの `.shape` と名称の組み合わせとして、**正しい**ものはどれですか？

- **a.** `[3]` は3次元テンソルであり、`[2, 3]` は2次元と3次元を組み合わせた合計5次元のテンソルである
- **b.** `[]`＝0次元（スカラー）、`[3]`＝1次元（ベクトル）、`[2, 3]`＝2次元（2行3列の行列）、`[1, 6, 151936]`＝3次元テンソル
- **c.** スカラー・ベクトル・行列はテンソルとはまったく別のデータ型であり、PyTorchでは `torch.Tensor` に入れられない
- **d.** `[2, 3]` は「3行2列の行列」を意味しており、中に含まれる要素の総数は足し算の $2 + 3 = 5$ 個である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：テンソルの次元数（ランク）は「`.shape` に並んでいる数字の個数」で決まり、2次元は `[行, 列]` の順です。
- **ヒント 2**：要素数は足し算ではなく各軸の掛け算です。0次元＝スカラー、1次元＝ベクトル、2次元＝行列の対応を確認しましょう。
- **正解**：`b`

**考え方**：テンソルの次元数は `.shape` に並ぶ整数の個数で決まり、0次元＝スカラー、1次元＝ベクトル、2次元＝行列、3次元以上も含めて総称してテンソルと呼びます。

1. `torch.tensor(5)` → shape `[]`（0次元スカラー）。
1. `torch.tensor([1, 2, 3])` → shape `[3]`（1次元ベクトル）。
1. `torch.tensor([[1, 2, 3], [4, 5, 6]])` → shape `[2, 3]`（2次元行列：2行×3列）。
1. 第3章の `out.logits` → shape `[1, 6, 151936]`（3次元テンソル）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q02

</details>

#### 📝 確認問題（`ch06-q03` · Lv.2 · 2次元テンソルのインデックスとスライス（M[1, 2], M[:, 1]））

2行3列の行列 `M = torch.tensor([[1, 2, 3], [4, 5, 6]])` に対して、`M[:, 1]` と `M[1, 2]` の組み合わせとして正しいものはどれですか？

- **a.** PyTorchの2次元テンソルでは `:` を使った列切り出しがエラーになり、`for` 文でしか取り出せない
- **b.** `M[:, 1]` は `tensor([2, 5])`（全行の1列目）、`M[1, 2]` は `tensor(6)`（1行目の2列目）
- **c.** `M[:, 1]` は `tensor([4, 5, 6])`（1行目すべて）、`M[1, 2]` は `tensor(2)`（0行目の1列目）
- **d.** `M[:, 1]` は `tensor([1, 4])`（1始まりで数えた1列目）、`M[1, 2]` は `tensor(5)`（中央の要素）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：PyTorchの添字は `M[行, 列]` の順で、どちらも **0始まり** です。`:` は「その軸のすべて」を表します。
- **ヒント 2**：行0が `[1, 2, 3]`、行1が `[4, 5, 6]` です。各行のインデックス1（2番目）と、行1のインデックス2（3番目）を取り出します。
- **正解**：`b`

**考え方**：`M` の0行目は `[1, 2, 3]`、1行目は `[4, 5, 6]` です。`M[:, 1]` は全行から1列目（2と5）を取り出し、`M[1, 2]` は1行目の2列目（`6`）を取り出します。

1. 行0：`[1, 2, 3]`（列0=`1`, 列1=`2`, 列2=`3`）。
1. 行1：`[4, 5, 6]`（列0=`4`, 列1=`5`, 列2=`6`）。
1. `M[:, 1]` → 全行の列1 → `tensor([2, 5])`。
1. `M[1, 2]` → 行1・列2 → `tensor(6)`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q03

</details>

## bigramの表を作る

いよいよ本題です。第4章の「今の文字（行）×次の文字（列）」の表を、2061×2061の行列として作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
n = int(len(text) * 0.9)
train_text, val_text = text[:n], text[n:]

pairs = Counter(zip(train_text, train_text[1:]))
N = torch.zeros((V, V))
for (a, b), count in pairs.items():
    N[stoi[a], stoi[b]] = count

print(N.shape)
print("「先」→「生」の回数：", N[stoi["先"], stoi["生"]].item())
print("表のマスの数：", N.numel(), " うち0でないマス：", (N > 0).sum().item())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2061, 2061])
「先」→「生」の回数： 597.0
表のマスの数： 4247721  うち0でないマス： 20395
```

</details>

> **🐍 Python・PyTorchのポイント：.item() と比較**
>
> `N[i, j]` の結果は「数が1つだけ入ったテンソル」です。`.item()` を付けると、ふつうのPythonの数として取り出せます。<br>
> `N > 0` は、全部のマスについて「0より大きい？」を調べた、True/Falseの表を作ります。それを `.sum()` すると、Trueの数（＝0でないマスの数）が数えられます。

約420万マスのうち、0でないのは約2万マス（0.5%以下）。第4章で話した「スパース（まばら）」が、数字で見えました。

<details><summary><b>基礎の補足：理解のための最小コード：`.shape`・`.dtype`・`.item()`・`.numel()`</b>（クリックで開く）</summary>

| 属性 / メソッド | 具体例と出力 | 型 | 役割 |
|---|---|---|---|
| `x.shape` | `torch.Size([2, 3])` | `torch.Size` | 各軸の長さ（形）。デバッグの最優先チェック項目 |
| `x.dtype` | `torch.float32` / `torch.int64` / `torch.bool` | `torch.dtype` | 中身の数値型（`torch.zeros` は既定で `float32`、整数リストは `int64`） |
| `x.numel()` | `6`（$2 \times 3$） | `int` | テンソル内の総要素数（number of elements） |
| `s.item()` | `tensor(597.0).item()` → `597.0` | `float` / `int` | **要素1個だけ**のテンソルから普通のPython数値を取り出す |

```python no-run
import torch
t_int = torch.tensor([1, 2, 3])          # dtype: torch.int64（ID・インデックス用）
t_flt = torch.tensor([1.0, 2.0, 3.0])    # dtype: torch.float32（確率・重み用）
mask = (t_int > 1)                       # tensor([False, True, True]), dtype: torch.bool
print(mask.sum(), mask.sum().item())     # tensor(2) 2
```

また、テンソルの総要素数を保ったまま形を変える操作に **`.view(...)`** と **`.reshape(...)`** があります（第9章以降で多用します）。

```python no-run
x = torch.arange(6).reshape(2, 3)        # 形 [6] → [2, 3]
xt = x.transpose(0, 1)                   # 軸を入れ替えて [3, 2]（メモリ上は不連続になる）
# xt.view(6)                             # RuntimeError! メモリが不連続だと .view() は失敗する
print(xt.reshape(6))                     # OK! 必要なら自動でコピーして [6] に変形
print(xt.contiguous().view(6))           # OK! 連続化してから .view() しても同じ
```

- **`.view(...)`**：メモリをコピーせず高速に形だけ変えるが、メモリが連続（contiguous）でないとエラーになる。
- **`.reshape(...)`**：連続なら `.view()` と同じ（コピーなし）、不連続なら自動でコピーして安全に変形してくれる。迷ったら `.reshape()` を使うと事故を防げます（[付録B](https://sakunokyumei.github.io/llm-research-textbook/b-python.html)にも整理あり）。

</details>

#### 📝 確認問題（`ch06-q04` · Lv.2 · .shape・.dtype・.item() と条件マスク (N > 0).sum()）

本文のコード `(N > 0).sum().item()` がやっている処理の正しい順番と、最終的な戻り値の型はどれですか？

- **a.** ① `N > 0` で `bool` 表を作る → ② `.sum()` で `True` を数える → ③ `.item()` でPythonの `int` を取り出す
- **b.** ① `N > 0` で正のマスにある元の回数を合計（約14.6万）し → ② `.item()` で画面表示用の文字列 `str` へ変換して返す
- **c.** ① `N > 0` で表の最大値が0より大きいかを判定する `True` 1個を返し → ② `.sum().item()` で必ず整数 `1` を返す
- **d.** ① `N > 0` で正の要素だけを抽出し → ② `.item()` でテンソル全体（420万要素）をPythonのリスト型へ一括変換する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：テンソルに対する比較 `N > 0` が各要素ごとに働くこと、そして `.item()` が何を返すメソッドか思い出しましょう。
- **ヒント 2**：`N > 0` は同じ形の `bool` テンソル、`.sum()` は `True=1` の総数（0次元テンソル）、`.item()` は1要素テンソルからPython数値を取り出します。
- **正解**：`a`

**考え方**：要素ごとの比較 `N > 0` で `True/False` のマスクを作り、`.sum()` で `True` の個数を集計、最後に `.item()` でスカラーテンソルからPythonの数値を取り出します。

1. `N > 0`：形 `[2061, 2061]`、`dtype=torch.bool`。
1. `.sum()`：`True=1, False=0` として合計 → `tensor(20395)`（形 `[]`、`dtype=torch.int64`）。
1. `.item()` → Pythonの `int` 型 `20395`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q04

</details>

## 回数を確率に変える：行ごとの合計とブロードキャスト

回数を確率にするには、各行を「その行の合計」で割ればよいのでした。スムージングの `k` も足しておきます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
k = 0.01
row_sums = (N + k).sum(dim=1, keepdim=True)
print(row_sums.shape)

P = (N + k) / row_sums
print(P.shape)
print("各行の合計（最初の5行）：", P.sum(dim=1)[:5])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2061, 1])
torch.Size([2061, 2061])
各行の合計（最初の5行）： tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
```

</details>

> **💡 理論のポイント：合計の方向と、ブロードキャスト**
>
> `sum(dim=1)` は「1番目の軸（横方向）に沿って足す」、つまり**各行の合計**です。`dim=0` なら各列の合計になります。`keepdim=True` は「足した軸を消さずに長さ1で残す」指定で、結果の形は `[2061, 1]`（縦に2061個並んだ数）になります。
>
> - `dim=0`：縦方向に足す（0番目の軸をつぶす → 各**列**の合計になる）
> - `dim=1`：横方向に足す（1番目の軸をつぶす → 各**行**の合計になる）
>
> 2行3列の具体例 $M = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}$（形 `[2, 3]`）で手計算してみましょう：
> - `M.sum(dim=0)` → 縦に足す：`[1+4, 2+5, 3+6] = [5, 7, 9]`（形 `[3]`）
> - `M.sum(dim=1)` → 横に足す：`[1+2+3, 4+5+6] = [6, 15]`（形 `[2]`）
> - `M.sum(dim=1, keepdim=True)` → 横に足し、軸1を長さ1で残す：`[[6], [15]]`（形 `[2, 1]`）
>
> そして `(N + k) / row_sums` では、形が `[2061, 2061]` と `[2061, 1]` で違うのに、割り算ができています。これは **ブロードキャスト** という仕組みで、長さ1の軸が自動的に「横にコピーされた」ものとして扱われるからです。
>
> <div class="math-display">[[a, b, c],      [[x],        [[a/x, b/x, c/x],
>  [d, e, f]]  ÷   [y]]    →    [d/y, e/y, f/y]]</div>
>
> 各行が、その行の合計で割られます。for文なしで、420万マスの割り算が一瞬です。

<details><summary><b>基礎の補足：ステップで理解するブロードキャストの「右揃えルール」</b>（クリックで開く）</summary>

2つのテンソルの形が違うとき、PyTorchは次の2ステップで計算できるか判定します。
1. **2つの形を右端にそろえて並べる**（足りない左側には `1` を補って考える）
2. **各軸について「数字が一致する」または「どちらかが `1`」ならOK**（`1` の側がもう片方の長さにコピーされる）

| 左の形 | 右の形 | 右揃えにした姿 | 判定と結果の形 |
|---|---|---|---|
| `[2, 3]` | `[2, 1]` | `(2, 3)` と `(2, 1)` | **OK** → `[2, 3]`（右の軸1が横に3倍コピー） |
| `[2, 3]` | `[3]` | `(2, 3)` と `(1, 3)` | **OK** → `[2, 3]`（右の軸0が縦に2倍コピー） |
| `[2, 3]` | `[2]` | `(2, 3)` と `(1, 2)` | **ERROR**（右端が `3` と `2` で不一致、かつどちらも `1` でない） |
| `[2, 1]` | `[1, 3]` | `(2, 1)` と `(1, 3)` | **OK** → `[2, 3]`（お互いに引き伸ばされる） |

</details>

#### ⚠️ つまずき注意：keepdim を忘れると「静かなバグ」が起きる！（2×2の具体例で実証）

`keepdim=True` を付けないと、`row_sums` の形は `[2061]`（1次元）になります。右揃えルールのため、正方形 `[2061, 2061]` とは次のように見なされます。

<div class="math-display">[2061, 2061] と [2061] → [2061, 2061] と [1, 2061] として扱われる</div>

小さな 2×2 行列 $N = \begin{pmatrix} 1 & 3 \\ 6 & 2 \end{pmatrix}$（各行の合計は 4 と 8）で、**正しい計算**と **`keepdim` 忘れのバグ** を手計算で比べてみましょう：

- **正しい計算（`keepdim=True`、形 `[2, 1]`＝`[[4], [8]]`）**：横にコピーされて `[[4, 4], [8, 8]]` で割る
  $$P = \begin{pmatrix} 1/4 & 3/4 \\ 6/8 & 2/8 \end{pmatrix} = \begin{pmatrix} 0.25 & 0.75 \\ 0.75 & 0.25 \end{pmatrix} \quad \text{（各行の合計：1.0, 1.0）}$$
- **バグ（`keepdim` 忘れ、形 `[2]` → `[1, 2]`＝`[[4, 8]]`）**：縦にコピーされて `[[4, 8], [4, 8]]` で**列ごと**に割ってしまう！
  $$\text{bad\_P} = \begin{pmatrix} 1/4 & 3/8 \\ 6/4 & 2/8 \end{pmatrix} = \begin{pmatrix} 0.25 & 0.375 \\ \mathbf{1.5} & 0.25 \end{pmatrix} \quad \text{（確率が1.5！各行の合計も 0.625, 1.75！）}$$

```python no-run
import torch
N_small = torch.tensor([[1., 3.], [6., 2.]])
print("正しい P:\n", N_small / N_small.sum(dim=1, keepdim=True))
print("バグ bad_P:\n", N_small / N_small.sum(dim=1))   # エラーにならず 1.5 が出る！
```

正方形行列では右端がたまたま一致するため、**エラーにならずに間違った答えが出ます**。形を `print` して `P.sum(dim=1)` が1になるか確かめるクセをつけましょう。

#### 📝 確認問題（`ch06-q05` · Lv.2 · sum(dim=0) と sum(dim=1) の手計算（2×3行列））

2行3列の行列 $M = \begin{pmatrix} 2 & 5 & 1 \\ 3 & 0 & 7 \end{pmatrix}$ に対して、`s = M.sum(dim=1)`（各**行**の合計）を計算したとき、**1番目の要素 `s[1]`**（下の行の合計）はいくつになりますか？（整数で入力）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`dim=1` は「1番目の軸（列方向＝横方向）をつぶして合計する」操作です。
- **ヒント 2**：0行目の合計が `s[0]`、1行目（下の行 `3, 0, 7`）の合計が `s[1]` になります。
- **正解**：`10`

**考え方**：`M.sum(dim=1)` は横方向に足して `tensor([8, 10])`（形 `[2]`）になるため、`s[1]` は $3 + 0 + 7 = 10$ です。

1. 軸1（横方向）に沿って合計する。
1. 行0：$2 + 5 + 1 = 8$。
1. 行1：$3 + 0 + 7 = 10$。
1. `M.sum(dim=1)` = `tensor([8, 10])` → `s[1] = 10`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q05

</details>

#### 📝 確認問題（`ch06-q06` · Lv.2 · N.sum(dim=0) と N.sum(dim=1, keepdim=True) の形と意味）

bigramの回数行列 `N`（形 `[2061, 2061]`、行＝今の文字、列＝次の文字）に対して、`N.sum(dim=0)` の**形**と**意味**の組み合わせとして正しいものはどれですか？

- **a.** 形は `[]`（0次元スカラー1個）。表全体の全マスを足し合わせた総ペア数（約14.6万）を表す
- **b.** 形は `[2061, 1]`。各**行**の合計であり、「それぞれの文字が**今の文字**として何回出たか」を表す
- **c.** 形は `[2061]`。各**列**の合計であり、「それぞれの文字が**次の文字**として何回出たか」を表す
- **d.** 形は `[1, 2061]`。各行の中で最も出現回数が多かった文字の最大出現回数を並べて表す

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`N` の軸0は「今の文字（行）」、軸1は「次の文字（列）」です。`sum(dim=0)` はどちらの軸をつぶして合計しますか？
- **ヒント 2**：軸0をつぶす（`keepdim` なし）と形は `[2061]` になり、列ごとに縦へ足すので「次の文字としての出現総数」が残ります。
- **正解**：`c`

**考え方**：`[2061, 2061]` の軸0（今の文字）に沿って合計すると、軸0が消えて形 `[2061]` になり、各要素 `j` は「どんな文字の次であってもとにかく文字 `j` が次の文字として出た総回数」になります。

1. 形：`[2061, 2061]` の軸0を `sum` でつぶす（`keepdim=False`）→ `[2061]`。
1. 計算：列 `j` を縦に全部足す $= \sum_i N[i, j]$。
1. 意味：文字 `j` が「次の文字」として登場した回数。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q06

</details>

*(この節の追加問題 `ch06-q07`, `ch06-q08`, `ch06-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html で自動採点・復習できます)*

## 損失を一瞬で計算する

第5章では、for文で1文字ずつ損失を計算しました。テンソルなら、こうなります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
val_ids = torch.tensor([stoi[c] for c in val_text])
xs, ys = val_ids[:-1], val_ids[1:]

start = time.time()
probs = P[xs, ys]
loss = -probs.log().mean()
print(f"検証データの損失：{loss.item():.3f}（{time.time() - start:.4f}秒）")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
検証データの損失：3.581（0.0012秒）
```

</details>

第5章と同じ **3.581** です。`P[xs, ys]` は「`xs` の各文字の行の、`ys` の各文字の列」をいっぺんに取り出す書き方（**高度なインデックス**）で、16,246個の確率が一度に手に入ります。`.log()` で全部の対数をとり、`.mean()` で平均。3行で損失が出ました。

**ボンゴ先生**：同じことをfor文でやると、パソコンにもよるが100倍くらい時間がかかる。テンソルの計算は、中身がC言語などで書かれた高速なプログラムで、しかもまとめて処理されるからだ。「Pythonでfor文を書いたら負け」という格言が、機械学習の世界にはある。……まあ、第4章で散々書いたけどね。

#### 📝 確認問題（`ch06-q10` · Lv.2 · val_ids[:-1], val_ids[1:] と高度なインデックス P[xs, ys]）

本文の `xs, ys = val_ids[:-1], val_ids[1:]` と `P[xs, ys]` の仕組みを純粋なPythonリストで再現します。次のコードを実行すると何が表示されますか？

```python
P = [[0.1, 0.9], [0.8, 0.2]]
ids = [0, 1, 0, 0]
xs, ys = ids[:-1], ids[1:]
probs = [P[x][y] for x, y in zip(xs, ys)]
print(probs)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`xs = ids[:-1]`（末尾を除く）と `ys = ids[1:]`（先頭を除く）を並べると、隣接する `(今のID, 次のID)` ペアの列ができます。
- **ヒント 2**：`xs = [0, 1, 0]`、`ys = [1, 0, 0]` なので、順に `P[0][1]`, `P[1][0]`, `P[0][0]` を読み取ってリストにします。
- **期待される出力**：`[0.9, 0.8, 0.1]`

**考え方**：隣接IDペア `(0→1), (1→0), (0→0)` に対応する `P[0][1]=0.9`, `P[1][0]=0.8`, `P[0][0]=0.1` が取り出されます。PyTorchではこれが `P[xs, ys]` の1式でfor文なしに完了します。

1. `xs = ids[:-1] = [0, 1, 0]`、`ys = ids[1:] = [1, 0, 0]`。
1. `P[0][1] = 0.9`、`P[1][0] = 0.8`、`P[0][0] = 0.1`。
1. 結果：`[0.9, 0.8, 0.1]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q10

</details>

## 表を眺める

表の一部を、色の濃さで描いてみましょう。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

look = ["私", "先", "生", "は", "の", "に", "を", "た", "。", "、", "奥", "さ", "ん", "Ｋ"]
idx = [stoi[c] for c in look]
sub = P[idx][:, idx]

plt.figure(figsize=(6, 5))
plt.imshow(sub, cmap="Blues")
plt.xticks(range(len(look)), look)
plt.yticks(range(len(look)), look)
plt.xlabel("次の文字")
plt.ylabel("今の文字")
plt.colorbar(label="確率")
plt.title("bigramの確率表（一部）")
plt.show()

![bigramの確率表の一部。先→生、奥→さ、さ→ん、私→は、。→私などが濃い](https://sakunokyumei.github.io/llm-research-textbook/img/figs/06-tensors/cell7_0.png)

「先→生」「奥→さ」「さ→ん」（奥さん！）が濃く光っています。この表こそが、bigramモデルの「頭の中」のすべてです。

## 表から文章を作る

テンソル版の生成も作っておきましょう。`torch.multinomial` は、第3章で自作したくじ引きのテンソル版です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
g = torch.Generator().manual_seed(0)
ix = stoi["私"]
out = ["私"]
for _ in range(60):
    ix = torch.multinomial(P[ix], num_samples=1, generator=g).item()
    out.append(itos[ix])
print("".join(out))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私は段加減です。そう渇様子先生といと私はた。しまで。私は一種を極ったがら見える資刹郵濯耐皇奪焼肩煎諺者櫓剛示樹息貪橋の男で
```

</details>

（出てくる文章は、乱数の種 `manual_seed(0)` の数字を変えると変わります。）

**ミナト**：途中で「資刹郵濯耐皇奪…」って、読めない漢字が延々と続いてますけど……。

**ボンゴ先生**：いいところに気づいた！ これは**スムージングの副作用**だ。珍しい文字の行は、数えた回数の合計がとても少ない。そこに全部の文字へ0.01ずつ、合計 $0.01 \times 2061 \approx 20.6$ 回分の「おまけ」を足すと、おまけのほうが本物の回数より多くなってしまう。するとその行は、ほとんど「でたらめ」になる。一度珍しい文字に迷い込むと、次も珍しい文字、その次も……と抜け出せなくなるんだ。数えるだけのモデルの、もう1つの限界だね。

#### 📝 確認問題（`ch06-q11` · Lv.3 · スムージングした表から生成すると珍しい漢字が連続する理由）

スムージング（`k=0.01`）を足した確率表 `P` から文章を生成したら、途中で「資刹郵濯耐皇奪…」と珍しい漢字が延々と続いてしまいました。この現象の正しい原因はどれですか？

- **a.** PyTorchの `torch.multinomial` は語彙番号の大きい漢字を優先的にサンプリングする仕様上の偏りを持っているから
- **b.** 低頻度文字の行はおまけ合計（約20.6）が本物の回数（1〜2回）を圧倒してほぼ一様ランダムになるから
- **c.** 乱数シードに `torch.manual_seed(0)` を指定すると、仕様上かならず珍しい漢字ばかり出力される乱数列になるから
- **d.** スムージング定数 `k=0.01` を足したあとに行ごとの正規化が崩れ、確率表 `P` の各行の合計が1.0を超えてしまっているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：本文に1〜2回しか登場しない珍しい漢字の行で、「本物の出現回数」と「全2061文字に足したおまけの合計」の比率を比べてみましょう。
- **ヒント 2**：おまけ合計 $0.01 \times 2061 \approx 20.6$ が本物の回数 `1` より20倍大きく、ほぼ完全ランダムな行になってしまいます。
- **正解**：`b`

**考え方**：珍しい文字の行では本物の情報（回数1）よりノイズのおまけ（合計20.6）が約20倍も大きくなり、ほぼ「2061文字から完全ランダム」に近づきます。日本語の語彙2061文字の大半は珍しい漢字なので、一度迷い込むと珍しい漢字→珍しい漢字のループに陥ります。

1. 出現1回の珍しい漢字の行：本物 `1` 回 vs おまけ `0.01 × 2061 = 20.61` 回。
1. 約95%の確率で「おまけ」から一様ランダムに次の文字が選ばれる。
1. 2061語の大半は低頻度漢字なので、次もまた珍しい漢字を引き続けてしまう。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q11

</details>

## one-hotと行列の掛け算

最後に、次の章から何度も出てくる大事な計算を1つ。**行列の掛け算**です。

文字「私」を、「私」の位置だけ1で、残りが全部0の長さ2061のベクトルで表すことにします。これを **one-hot**（ワンホット、1か所だけ熱い）ベクトルと言います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch.nn.functional as F

x = F.one_hot(torch.tensor(stoi["私"]), num_classes=V).float()
print(x.shape, x.sum())

row = x @ P            # one-hot × 確率表
print(row.shape)
print(torch.allclose(row, P[stoi["私"]]))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([2061]) tensor(1.)
torch.Size([2061])
True
```

</details>

`@` は行列の掛け算の記号です。one-hotベクトルに表を掛けると、**表の「私」の行がそのまま取り出される**ことが確かめられました（`allclose` は「ほぼ等しい？」を調べる関数）。

#### 📐 数学のポイント：行列の掛け算 `@`：9ステップで理由から理解する

**① 解決したい問題**：「入力の重み付き和」や「表からの行の取り出し」を、1件ずつではなく何千件分も一気に・微分（第7章）可能な数式で計算したい。

**② 要素ごとの積 `*` と行列積 `@` の決定的な違い**：
- `A * B`：同じ位置のマスどうしを掛けるだけ（形が一致するかブロードキャストが必要）。
- `A @ B`：左の**行**と右の**列**の同じ位置どうしを掛けて足す（**内積**を全ペアで計算する）。

**③ 小さな例で手計算（`1×2 @ 2×3` と `2×2 @ 2×2`）**：
まず `1×2` と `2×3` の積です。

$$\begin{pmatrix} 1 & 2 \end{pmatrix} \begin{pmatrix} 3 & 4 & 5 \\ 6 & 7 & 8 \end{pmatrix} = \begin{pmatrix} 1\cdot3+2\cdot6 & 1\cdot4+2\cdot7 & 1\cdot5+2\cdot8 \end{pmatrix} = \begin{pmatrix} 15 & 18 & 21 \end{pmatrix}$$

2行2列どうしでも同じ手順です（左の `i` 行目 × 右の `j` 列目 ＝ 結果の `(i, j)` マス）：
$$\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix} \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix} = \begin{pmatrix} 1\cdot5+2\cdot7 & 1\cdot6+2\cdot8 \\ 3\cdot5+4\cdot7 & 3\cdot6+4\cdot8 \end{pmatrix} = \begin{pmatrix} 19 & 22 \\ 43 & 50 \end{pmatrix}$$

左の**行**と右の**列**の、同じ位置どうしを掛けて足す。これをすべての組み合わせでやります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
A = torch.tensor([[1., 2.]])
B = torch.tensor([[3., 4., 5.],
                  [6., 7., 8.]])
print(A @ B)
print(A.shape, "@", B.shape, "→", (A @ B).shape)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[15., 18., 21.]])
torch.Size([1, 2]) @ torch.Size([2, 3]) → torch.Size([1, 3])
```

</details>

**④ 一般式と形のルール**：
$$(AB)_{i, j} = \sum_{t=0}^{k-1} A_{i, t}\, B_{t, j}$$
- **形のルール**：$(n \times k)$ の行列と $(k \times m)$ の行列を掛けると、$(n \times m)$ になります。**真ん中の $k$ が一致していないと掛けられません**（例：`[2, 3] @ [2, 3]` はエラー、`[32, 2061] @ [2061, 64]` は `[32, 64]`）。これが「形が合わない」エラーの正体です。

**⑤ なぜ `one-hot × 行列 ＝ 行の取り出し` になるのか？（具体計算）**：
$$\begin{pmatrix} 0 & 1 & 0 \end{pmatrix} \begin{pmatrix} 10 & 20 \\ 30 & 40 \\ 50 & 60 \end{pmatrix} = 0\cdot\begin{pmatrix} 10 & 20 \end{pmatrix} + 1\cdot\begin{pmatrix} 30 & 40 \end{pmatrix} + 0\cdot\begin{pmatrix} 50 & 60 \end{pmatrix} = \begin{pmatrix} 30 & 40 \end{pmatrix}$$
one-hotベクトルを掛けると、「1の位置の行 × 1」以外は全部「× 0」で消えるので、1の位置の行がそのまま出てくるのです。

**⑥ 純粋なPythonの3重ループとの対応**：
```python
# (n x k) @ (k x m) -> (n x m) の正体は、実はこの3重ループ！
C = [[sum(A[i][t] * B[t][j] for t in range(k)) for j in range(m)] for i in range(n)]
```

> **💡 理論のポイント：なぜこれが大事なのか**
>
> 「one-hot × 表 ＝ 表の行を取り出す」。当たり前に見えますが、ここに大事な発想の転換があります。
>
> - 今は、表の中身（確率）を**数えて**決めました
> - でも、表の中身を**学習で少しずつ調整する数（パラメータ）**にしたら？
>
> 第8章では、まさにそれをやります。表の中身を最初はでたらめにして、損失が小さくなるように少しずつ調整していくのです。そのためには「調整の方向」を知る必要があります。それが次の章の **微分** です。

#### 📝 確認問題（`ch06-q12` · Lv.2 · 行列の掛け算 @ の形のルール (n×k) @ (k×m) = (n×m)）

形が `[32, 2061]` のテンソルと `[2061, 64]` のテンソルを行列積 `@` で掛けると、結果の形は何になりますか？

- **a.** `[32, 64]`（内側の2061が一致して消え、外側の32と64が残る）
- **b.** `[2061, 2061]`（一致した内側の軸同士がそのまま結果の形になる）
- **c.** 2つのテンソルの形が完全一致していないため行列積はエラーになる
- **d.** `[32, 2061, 64]`（3つの数値がそのまま連結されて3次元テンソルになる）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：行列積 `@` の形のルール $(n \times k) @ (k \times m) = (n \times m)$ を思い出しましょう。
- **ヒント 2**：左の列数と右の行数（真ん中の `2061`）が一致して消え、外側の軸が残ります。
- **正解**：`a`

**考え方**：`[32, 2061] @ [2061, 64]` は内側の `2061` 同士が一致しているため計算可能で、外側の `[32, 64]` が残ります。

1. 左の形：`[32, 2061]`（$n=32, k=2061$）。
1. 右の形：`[2061, 64]`（$k=2061, m=64$）。
1. 真ん中の `2061` が一致 → 結果の形は `[32, 64]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q12

</details>

#### 📝 確認問題（`ch06-q13` · Lv.3 · 行列の掛け算 @ の手計算（2×2、1×2 @ 2×3））

2×2行列 $A = \begin{pmatrix} 2 & 1 \\ 0 & 3 \end{pmatrix}$ と $B = \begin{pmatrix} 4 & 5 \\ 2 & 6 \end{pmatrix}$ の行列積 $C = A @ B$ を手計算します。**0行1列目の要素 `C[0, 1]`**（右上）はいくつになりますか？（整数で入力）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：行列積の `(i, j)` 成分は「左の行列 $A$ の `i` 行目」と「右の行列 $B$ の `j` 列目」の内積（同じ位置を掛けて足す）です。
- **ヒント 2**：`C[0, 1]` は $A$ の0行目 `[2, 1]` と $B$ の1列目 `[5, 6]` の同じ位置どうしを掛けて足します。
- **正解**：`16`

**考え方**：$C_{0, 1} = 2 \times 5 + 1 \times 6 = 10 + 6 = 16$ です。

1. $A$ の0行目：`[2, 1]`。
1. $B$ の1列目：`[5, 6]`。
1. 掛けて足す：$2 \times 5 + 1 \times 6 = 10 + 6 = 16$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q13

</details>

*(この節の追加問題 `ch06-q14`, `ch06-q15` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html で自動採点・復習できます)*

## GPUは何が速いのか

#### 💡 理論のポイント：並列計算

行列の掛け算は、「掛けて足す」をたくさん、**お互いに関係なく**計算できます。2061×2061の行列どうしの掛け算なら、約87億回の掛け算です。

- **CPU**：とても賢い職人が数人〜数十人。複雑な仕事が得意
- **GPU**：そこそこの作業員が数千人。単純な計算を**同時に**こなすのが得意

行列の掛け算は「単純な計算を大量に」の典型なので、GPUの独壇場です。LLMの計算のほとんどは行列の掛け算なので、LLMにはGPUが欠かせないのです。Colabで確かめたい人は、次を実行してみてください（CPUのままなら `cuda` の行は飛ばされます）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
X = torch.randn(2000, 2000)
start = time.time(); _ = X @ X; print(f"CPU：{time.time() - start:.3f}秒")
if torch.cuda.is_available():
    Xg = X.to("cuda"); _ = Xg @ Xg; torch.cuda.synchronize()
    start = time.time(); _ = Xg @ Xg; torch.cuda.synchronize()
    print(f"GPU：{time.time() - start:.4f}秒")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
CPU：0.152秒
GPU：0.0021秒
```

</details>

#### 📝 確認問題（`ch06-q16` · Lv.2 · テンソル計算とGPUが行列の掛け算で圧倒的に速い理由）

Pythonの `for` 文で1文字ずつ計算するより PyTorch のテンソル演算が速く、さらに巨大な行列積 `X @ X` ではCPUよりGPUが数十〜数百倍速いのはなぜですか？ **Pythonループとの違い** と **行列積の並列性（CPU vs GPU）** の2点から説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「Pythonのfor文 vs PyTorchテンソル演算」と「CPU vs GPU の行列積」の2段階に分けて考えましょう。
- **ヒント 2**：① テンソルは内部の高速なネイティブコードで一括処理する点、② 行列積の各出力マスは独立なので数千コアのGPUで同時並列計算できる点、の2点をまとめます。
- **解答の目安**：① PyTorchのテンソル演算はPythonの遅いfor文を1回ずつ解釈せず、内部のC/C++などで最適化された高速プログラムで全要素をまとめて処理するため速い。② 行列の掛け算の各マス $(i, j)$ の「掛けて足す」計算はお互いに独立しているため、少数の高性能コアで順番にこなすCPUよりも、数千個のコアで単純計算を同時に並列実行できるGPUの方が圧倒的に速い。

**考え方**：LLMの計算の大部分は巨大な行列の掛け算です。だからこそ、テンソルで並列化できる形に数式を書き直し、GPUで一気に計算する設計が現代AIの土台になっています。

1. Pythonの `for` 文 vs テンソル：1要素ごとのインタプリタ処理を避け、最適化済みネイティブコードで一括実行。
1. CPU vs GPU：行列積の各出力要素は互いに依存しないため、数千の単純コアで同時並列計算するGPUに完璧にマッチする。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html#ex-ch06-q16

</details>

<details><summary><b>もっと深く：NumPyとの関係</b>（クリックで開く）</summary>

Pythonで数の表を扱う道具としては、**NumPy**（ナムパイ）も有名です。`numpy.array` はテンソルとほぼ同じ使い心地で、データ分析ではNumPyが主役です。PyTorchのテンソルはNumPyの配列に「GPUで動く」「微分を自動で計算できる（次の章）」機能を足したもの、と考えてOKです。`t.numpy()` と `torch.from_numpy(a)` で行き来できます。

</details>

**✅ できたらチェック**

- [ ] テンソルの足し算・掛け算がfor文なしでできることを確かめた
- [ ] `.shape` で形を確認するクセがついた
- [ ] bigramの表を行列 `N` として作り、確率の表 `P` に変換した
- [ ] テンソルで損失を計算し、第5章と同じ3.581になった
- [ ] one-hot × 行列 ＝ 行の取り出し、を確かめた

> **📝 まとめ**
>
> - **テンソル**は数を並べた箱。1次元がベクトル、2次元が行列。一番大事なのは **形（shape）**
> - テンソルの計算はfor文なしで全要素をまとめて処理するので、とても速い
> - `sum(dim=1, keepdim=True)` と **ブロードキャスト** で、各行を合計で割れる
> - 行列の掛け算の形のルール：$(n \times k) @ (k \times m) = (n \times m)$
> - **one-hot × 表 ＝ 表の行の取り出し**。この表の中身を「学習で決める」のが次の目標

**ボンゴ先生**：表の中身を「数える」のではなく「学習で決める」。そのためには、中身を少し変えたら損失がどう変わるかを知る必要がある。それを教えてくれるのが**微分**だ。次の章は、坂道を転がるボールの話から始めよう。物理屋の血が騒ぐね。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/06-tensors.html
- **次の章**：[第7章 学習とは坂を下ること：微分と勾配降下法](https://sakunokyumei.github.io/llm-research-textbook/07-gradient.html)